In [3]:
# !pip install yfinance


  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.4-py3-none-any.whl.metadata (8.6 kB)
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached websockets-16.0-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
  Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/133.7 kB ? eta -:--:--
   --------- ------------------------------ 30.7/133.7 kB 1.3 MB/s eta 0:00:01
   -------------------- ------------------ 71.7/133.7 kB 975.2 kB/s eta 0:00:01
   ---------------------------------------- 133.7/133.7 kB 1.1 MB/s eta 0:00:00
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached frozendict-2.4.7-py3-none-any.whl (16 kB)
Using cached peewee-4.0.4-py3-none-any.whl (144 kB)
Using cached websockets-16.0-cp311-cp311-win_amd64.whl (178 kB)
Using cached cffi-2.0.0-cp311-cp311-win_amd64.whl (182 

  You can safely remove it manually.


In [13]:
import yfinance as yf
import json
from datetime import datetime, timezone

In [8]:
ticker = yf.Ticker("PETR4.SA")

empresa = ticker.info
noticias = ticker.get_news(count=5, tab="news")


In [9]:
print(empresa)


{'address1': 'Avenida Henrique Valadares, 28', 'city': 'Rio De Janeiro', 'state': 'RJ', 'zip': '20231-030', 'country': 'Brazil', 'phone': '55 21 3224 2401', 'website': 'https://petrobras.com.br', 'industry': 'Oil & Gas Integrated', 'industryKey': 'oil-gas-integrated', 'industryDisp': 'Oil & Gas Integrated', 'sector': 'Energy', 'sectorKey': 'energy', 'sectorDisp': 'Energy', 'longBusinessSummary': 'Petróleo Brasileiro S.A. - Petrobras explores, produces, and sells oil and gas in Brazil and internationally. It operates through three segments: Exploration and Production; Refining, Transportation & Marketing; and Gas & Low Carbon Energies. The Exploration and Production segment explores, develops, and produces crude oil, natural gas liquids, and natural gas primarily for supplies to the domestic refineries. The Refining, Transportation and Marketing segment engages in the refining, logistics, transport, acquisition, and export of crude oil; trading of oil products; and production of fertili

In [10]:
print(noticias)

[{'id': '1a36d27b-3743-3c14-8d8f-12b90a2216c5', 'content': {'id': '1a36d27b-3743-3c14-8d8f-12b90a2216c5', 'contentType': 'STORY', 'title': '4 Undervalued PEG Stocks With Strong Earnings Growth Outlook', 'description': '', 'summary': 'SNX and three peers stand out as undervalued PEG plays, combining low valuations with solid earnings growth potential amid volatile markets.', 'pubDate': '2026-04-17T19:00:00Z', 'displayTime': '2026-04-17T19:00:00Z', 'isHosted': True, 'bypassModal': False, 'previewUrl': None, 'thumbnail': None, 'provider': {'displayName': 'Zacks', 'url': 'http://www.zacks.com/', 'sourceId': 'zacks.com'}, 'canonicalUrl': {'url': 'https://finance.yahoo.com/markets/stocks/articles/4-undervalued-peg-stocks-strong-190000841.html', 'site': 'finance', 'region': 'US', 'lang': 'en-US'}, 'clickThroughUrl': {'url': 'https://finance.yahoo.com/markets/stocks/articles/4-undervalued-peg-stocks-strong-190000841.html', 'site': 'finance', 'region': 'US', 'lang': 'en-US'}, 'metadata': {'edit

In [16]:
company_payload = {
    "ticker": empresa.get("symbol"),
    "name": empresa.get("longName") or empresa.get("shortName"),
    "sector": empresa.get("sector"),
    "industry": empresa.get("industry"),
    "country": empresa.get("country"),
    "description": empresa.get("longBusinessSummary"),
    "extra": {
        "marketCap": empresa.get("marketCap"),
        "currency": empresa.get("currency")
    }
}

primeira_noticia = noticias[0]
content = primeira_noticia.get("content", {})

# Trata data
pub_date = content.get("pubDate")
published_at_iso = None

if pub_date:
    if isinstance(pub_date, (int, float)):
        published_at_iso = datetime.fromtimestamp(pub_date, tz=timezone.utc).isoformat().replace("+00:00", "Z")
    elif isinstance(pub_date, str):
        try:
            published_at_iso = datetime.fromisoformat(pub_date.replace("Z", "+00:00")).isoformat().replace("+00:00", "Z")
        except ValueError:
            if pub_date.isdigit():
                published_at_iso = datetime.fromtimestamp(int(pub_date), tz=timezone.utc).isoformat().replace("+00:00", "Z")
            else:
                published_at_iso = pub_date

# Trata publisher/provider
provider = content.get("provider")

if isinstance(provider, dict):
    publisher_value = provider.get("displayName") or provider.get("sourceId") or str(provider)
else:
    publisher_value = provider

# Trata URL
canonical_url = content.get("canonicalUrl")
click_url = content.get("clickThroughUrl")

url_value = None
if isinstance(canonical_url, dict):
    url_value = canonical_url.get("url")
if not url_value and isinstance(click_url, dict):
    url_value = click_url.get("url")

news_payload = {
    "title": content.get("title"),
    "summary": content.get("summary"),
    "publisher": publisher_value,
    "published_at": published_at_iso,
    "url": url_value,
    "extra": {}
}

payload_requisicao = {
    "company": company_payload,
    "news": news_payload
}

print(json.dumps(payload_requisicao, indent=2, ensure_ascii=False))

{
  "company": {
    "ticker": "PETR4.SA",
    "name": "Petróleo Brasileiro S.A. - Petrobras",
    "sector": "Energy",
    "industry": "Oil & Gas Integrated",
    "country": "Brazil",
    "description": "Petróleo Brasileiro S.A. - Petrobras explores, produces, and sells oil and gas in Brazil and internationally. It operates through three segments: Exploration and Production; Refining, Transportation & Marketing; and Gas & Low Carbon Energies. The Exploration and Production segment explores, develops, and produces crude oil, natural gas liquids, and natural gas primarily for supplies to the domestic refineries. The Refining, Transportation and Marketing segment engages in the refining, logistics, transport, acquisition, and export of crude oil; trading of oil products; and production of fertilizers, as well as holding interests in petrochemical companies. The Gas and Low Carbon Energies segment is involved in the logistic and trading of natural gas and electricity; transportation and tr